# Tinker RL Fine-Tuning on GSM8K Math

This notebook demonstrates **reinforcement learning fine-tuning** of a small language model (`Qwen/Qwen3.5-4B`) using [Tinker](https://tinker.ai/) and compares it against a Cortex `llama3.3-70b` baseline on the GSM8K grade-school math benchmark.

**Key question:** Can targeted RL training close the gap between a 4B-parameter model and a 70B general-purpose model on math reasoning?

| Component | Details |
|---|---|
| **Fine-tuned model** | Qwen/Qwen3.5-4B via Tinker GRPO |
| **Baseline model** | Cortex llama3.3-70b |
| **Benchmark** | GSM8K (50 eval problems) |
| **Judge** | Cortex mistral-large2 |

## 1. Setup
Install dependencies, import libraries, and configure hyperparameters.

In [ ]:
!pip install tinker-cookbook datasets --quiet

In [ ]:
!pip install tinker

In [ ]:
import os
import re
import json
import time
import logging
from concurrent.futures import Future

import pandas as pd
import numpy as np
import datasets
import tinker
import torch
from tinker import types
from tinker.types.tensor_data import TensorData

from tinker_cookbook import model_info, renderers
from tinker_cookbook.tokenizer_utils import get_tokenizer

from snowflake.snowpark.context import get_active_session
import snowflake.cortex as cortex

session = get_active_session()
print("Connected to Snowflake")

In [ ]:
os.environ["TINKER_API_KEY"] = ""  # <-- paste your key

TINKER_MODEL   = "Qwen/Qwen3.5-4B"
LORA_RANK      = 32
LEARNING_RATE  = 1e-4
GROUP_SIZE     = 4        # rollouts per problem
BATCH_SIZE     = 100      # problems per batch
TRAIN_STEPS    = 10       # batches to train (increase for better results)
MAX_TOKENS     = 256

# --- Cortex baseline ---
CORTEX_MODEL   = "llama3.3-70b"

### Load GSM8K Dataset
Load the [GSM8K](https://huggingface.co/datasets/openai/gsm8k) grade-school math benchmark. We use the full training set for RL and 50 test problems for evaluation.

In [ ]:
ds = datasets.load_dataset("openai/gsm8k", "main")
train_data = ds["train"]
test_data  = ds["test"].select(range(50))  # 50 problems for eval

print(f"Train: {len(train_data)} | Eval: {len(test_data)}")
print(f"\nSample problem:\n{test_data[0]['question']}")
print(f"\nAnswer:\n{test_data[0]['answer']}")

### Grading Utilities
Helper functions to extract and compare numerical answers from model responses. Supports `\boxed{}` format, `####` GSM8K format, and fallback to last-number extraction.

In [ ]:
def extract_gsm8k_answer(answer_text: str) -> str:
    """Extract the number after #### in GSM8K ground truth."""
    match = re.search(r"####\s*(.+)", answer_text)
    if match:
        return match.group(1).strip().replace(",", "")
    return ""


def extract_boxed(text: str) -> str:
    """Extract content from \\boxed{...}."""
    match = re.search(r"\\boxed\{([^}]+)\}", text)
    if match:
        return match.group(1).strip().replace(",", "")
    return ""


def extract_any_number(text: str) -> str:
    """Fallback: grab the last number in the response."""
    numbers = re.findall(r"-?[\d,]+\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "")
    return ""


def grade(response: str, ground_truth: str) -> bool:
    """Check if the model's answer matches ground truth."""
    gt = extract_gsm8k_answer(ground_truth)
    # Try \boxed{} first, then last number
    pred = extract_boxed(response) or extract_any_number(response)
    try:
        return abs(float(pred) - float(gt)) < 1e-3
    except (ValueError, TypeError):
        return pred.strip() == gt.strip()


# Quick sanity check
assert extract_gsm8k_answer("some work\n#### 42") == "42"
assert extract_boxed("The answer is \\boxed{42}") == "42"
print("Grading utilities ready")

## 2. Cortex Baseline
Run the Cortex `llama3.3-70b` model on 50 GSM8K eval problems to establish a baseline accuracy before fine-tuning.

In [ ]:
MATH_SYSTEM_PROMPT = (
    "Solve the math problem step by step. "
    "Put your final numerical answer inside \\boxed{}."
)


def cortex_baseline(question: str) -> str:
    """Call Cortex COMPLETE for a single math question via SQL."""
    prompt = f"{MATH_SYSTEM_PROMPT}\n\nQuestion: {question}"
    escaped = prompt.replace("'", "''")
    result = session.sql(
        f"SELECT SNOWFLAKE.CORTEX.COMPLETE('{CORTEX_MODEL}', '{escaped}') AS resp"
    ).collect()
    return result[0]["RESP"]


# Run baseline on eval set
baseline_results = []
for i, row in enumerate(test_data):
    response = cortex_baseline(row["question"])
    correct  = grade(response, row["answer"])
    baseline_results.append({
        "idx": i,
        "question": row["question"][:80],
        "ground_truth": extract_gsm8k_answer(row["answer"]),
        "model_answer": extract_boxed(response) or extract_any_number(response),
        "correct": correct,
        "response": response,
    })
    if (i + 1) % 10 == 0:
        acc = sum(r["correct"] for r in baseline_results) / len(baseline_results)
        print(f"  [{i+1}/{len(test_data)}] running accuracy: {acc:.1%}")

baseline_acc = sum(r["correct"] for r in baseline_results) / len(baseline_results)
print(f"\nCortex {CORTEX_MODEL} baseline accuracy: {baseline_acc:.1%}")

## 3. Tinker RL Fine-Tuning
Create a Tinker training client, run GRPO (Group Relative Policy Optimization) on GSM8K training problems, and plot the reward curve.

**How it works:** For each batch of math problems, the model generates multiple rollouts. Correct answers get reward 1.0, wrong answers get 0.0. Advantages are computed relative to the group mean, and the model is updated via importance-sampled policy gradients.

In [ ]:
# Create Tinker training client
service_client  = tinker.ServiceClient()
training_client = service_client.create_lora_training_client(
    base_model=TINKER_MODEL, rank=LORA_RANK
)

tokenizer = get_tokenizer(TINKER_MODEL)
renderer_name = model_info.get_recommended_renderer_name(TINKER_MODEL)
renderer = renderers.get_renderer(renderer_name, tokenizer)

sampling_params = tinker.types.SamplingParams(
    max_tokens=MAX_TOKENS,
    stop=renderer.get_stop_sequences(),
)
adam_params = types.AdamParams(
    learning_rate=LEARNING_RATE, beta1=0.9, beta2=0.95, eps=1e-8
)

print(f"Tinker client ready — {TINKER_MODEL}, LoRA rank {LORA_RANK}")

In [ ]:
# --- Conversation template ---
CONVO_PREFIX = [{"role": "system", "content": MATH_SYSTEM_PROMPT}]
QUESTION_SUFFIX = "\nPlease reason step by step and put your final answer in \\boxed{}."

# --- RL Training Loop ---
train_metrics = []

for step in range(TRAIN_STEPS):
    t0 = time.time()

    # Grab a batch of training problems
    start = step * BATCH_SIZE
    end   = min(start + BATCH_SIZE, len(train_data))
    batch = train_data.select(range(start, end))

    # Get current model for sampling
    sampling_client = training_client.save_weights_and_get_sampling_client()

    # --- Rollouts ---
    futures, prompts = [], []
    for question in batch["question"]:
        convo = [
            *CONVO_PREFIX,
            {"role": "user", "content": question + QUESTION_SUFFIX},
        ]
        model_input = renderer.build_generation_prompt(convo)
        future = sampling_client.sample(
            prompt=model_input,
            num_samples=GROUP_SIZE,
            sampling_params=sampling_params,
        )
        futures.append(future)
        prompts.append(model_input)

    # --- Collect rewards and build training datums ---
    datums = []
    rewards_per_problem = []

    for future, prompt, answer in zip(futures, prompts, batch["answer"]):
        result = future.result()
        rewards_g, tokens_g, logprobs_g = [], [], []

        for seq in result.sequences:
            tokens_g.append(seq.tokens)
            logprobs_g.append(seq.logprobs)
            parsed_msg, _ = renderer.parse_response(seq.tokens)
            content = renderers.get_text_content(parsed_msg)
            gt = extract_gsm8k_answer(answer)
            pred = extract_boxed(content) or extract_any_number(content)
            try:
                reward = 1.0 if abs(float(pred) - float(gt)) < 1e-3 else 0.0
            except (ValueError, TypeError):
                reward = 0.0
            rewards_g.append(reward)

        mean_reward = sum(rewards_g) / len(rewards_g)
        advantages  = [r - mean_reward for r in rewards_g]
        rewards_per_problem.append(mean_reward)

        if all(a == 0.0 for a in advantages):
            continue  # skip if all rollouts agree

        ob_len = prompt.length - 1
        for toks, lps, adv in zip(tokens_g, logprobs_g, advantages):
            mi = prompt.append(types.EncodedTextChunk(tokens=toks[:-1]))
            target  = [0] * ob_len + toks
            pad_lp  = [0.0] * ob_len + lps
            pad_adv = [0.0] * ob_len + [adv] * (mi.length - ob_len)
            datums.append(types.Datum(
                model_input=mi,
                loss_fn_inputs={
                    "target_tokens": TensorData.from_torch(torch.tensor(target)),
                    "logprobs":      TensorData.from_torch(torch.tensor(pad_lp)),
                    "advantages":    TensorData.from_torch(torch.tensor(pad_adv)),
                },
            ))

    # --- Gradient step ---
    if datums:
        fwd_bwd = training_client.forward_backward(datums, loss_fn="importance_sampling")
        optim   = training_client.optim_step(adam_params)
        fwd_bwd.result()
        optim.result()

    batch_reward = sum(rewards_per_problem) / len(rewards_per_problem)
    elapsed = time.time() - t0
    train_metrics.append({"step": step, "reward": batch_reward, "time_s": elapsed})
    print(f"Step {step:3d} | reward {batch_reward:.3f} | datums {len(datums):4d} | {elapsed:.1f}s")

print("\nTraining complete")

In [ ]:
# Plot training reward curve
import matplotlib.pyplot as plt

df_train = pd.DataFrame(train_metrics)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_train["step"], df_train["reward"], marker="o")
ax.set_xlabel("Training Step")
ax.set_ylabel("Mean Reward")
ax.set_title(f"Tinker RL Training — {TINKER_MODEL}")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Evaluate Fine-tuned Model
Sample from the trained model on the same 50-problem eval set and grade the results.

In [ ]:
# Get sampling client from trained weights
finetuned_sampler = training_client.save_weights_and_get_sampling_client(
    name="math-rl-finetuned"
)

finetuned_results = []
eval_futures = []

for row in test_data:
    convo = [
        *CONVO_PREFIX,
        {"role": "user", "content": row["question"] + QUESTION_SUFFIX},
    ]
    model_input = renderer.build_generation_prompt(convo)
    future = finetuned_sampler.sample(
        prompt=model_input,
        num_samples=1,
        sampling_params=sampling_params,
    )
    eval_futures.append((future, row))

for i, (future, row) in enumerate(eval_futures):
    result = future.result()
    seq = result.sequences[0]
    parsed_msg, _ = renderer.parse_response(seq.tokens)
    response = renderers.get_text_content(parsed_msg)
    correct  = grade(response, row["answer"])

    finetuned_results.append({
        "idx": i,
        "question": row["question"][:80],
        "ground_truth": extract_gsm8k_answer(row["answer"]),
        "model_answer": extract_boxed(response) or extract_any_number(response),
        "correct": correct,
        "response": response,
    })
    if (i + 1) % 10 == 0:
        acc = sum(r["correct"] for r in finetuned_results) / len(finetuned_results)
        print(f"  [{i+1}/{len(test_data)}] running accuracy: {acc:.1%}")

finetuned_acc = sum(r["correct"] for r in finetuned_results) / len(finetuned_results)
print(f"\nFine-tuned {TINKER_MODEL} accuracy: {finetuned_acc:.1%}")

## 4. Compare Results
Build a side-by-side comparison DataFrame, visualize accuracy, and persist results to Snowflake.

In [ ]:
# Build comparison DataFrame
comparison = []
for b, f in zip(baseline_results, finetuned_results):
    comparison.append({
        "question": b["question"],
        "ground_truth": b["ground_truth"],
        f"cortex_{CORTEX_MODEL}_answer": b["model_answer"],
        f"cortex_{CORTEX_MODEL}_correct": b["correct"],
        "tinker_finetuned_answer": f["model_answer"],
        "tinker_finetuned_correct": f["correct"],
    })

df = pd.DataFrame(comparison)

print("=" * 60)
print(f"  Cortex {CORTEX_MODEL} (base):     {baseline_acc:.1%}")
print(f"  Tinker {TINKER_MODEL} (RL-tuned): {finetuned_acc:.1%}")
print(f"  Delta:                             {finetuned_acc - baseline_acc:+.1%}")
print("=" * 60)

# Show problems where results differ
differ = df[
    df[f"cortex_{CORTEX_MODEL}_correct"] != df["tinker_finetuned_correct"]
]
print(f"\n{len(differ)} problems where models disagree:")
df

In [ ]:
# Accuracy bar chart
import matplotlib.pyplot as plt

labels = [f"Cortex\n{CORTEX_MODEL}\n(base)", f"Tinker\n{TINKER_MODEL}\n(RL-tuned)"]
accs   = [baseline_acc, finetuned_acc]
colors = ["#29B5E8", "#FF6F00"]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, accs, color=colors, width=0.5, edgecolor="white")
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{acc:.1%}", ha="center", va="bottom", fontsize=14, fontweight="bold")

ax.set_ylabel("GSM8K Accuracy")
ax.set_title("Base vs RL Fine-Tuned — Math Reasoning")
ax.set_ylim(0, 1.15)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Write comparison results to a Snowflake table
session.sql("USE DATABASE CORTEX_CODE").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS CORTEX_CODE.PUBLIC").collect()
session.sql("USE SCHEMA CORTEX_CODE.PUBLIC").collect()

sp_df = session.create_dataframe(df)
sp_df.write.mode("overwrite").save_as_table("TINKER_VS_CORTEX_MATH_EVAL")
print("Results saved to CORTEX_CODE.PUBLIC.TINKER_VS_CORTEX_MATH_EVAL")

## 5. LLM-as-Judge Evaluation
Use Cortex `mistral-large2` as an impartial judge to score both models on **answer correctness** (0-5) and **reasoning quality** (0-5) for every problem.

In [ ]:
# ── LLM-as-Judge: score both models with Cortex ──

JUDGE_MODEL = "mistral-large2"

JUDGE_PROMPT = """You are an expert math evaluator. Score the model response.

**Question:** {question}
**Ground Truth Answer:** {ground_truth}
**Model Response:** {response}

Score on two dimensions (0-5 each):
1. **answer_correctness**: Does the final numerical answer match ground truth? 5=exact match, 3=close, 0=wrong
2. **reasoning_quality**: Is the step-by-step reasoning clear, correct, complete? 5=excellent, 0=absent

Return ONLY valid JSON:
{{"answer_correctness": <int>, "reasoning_quality": <int>, "reasoning": "<one sentence>"}}"""


def llm_judge(question: str, ground_truth: str, response: str) -> dict:
    prompt = JUDGE_PROMPT.format(
        question=question, ground_truth=ground_truth, response=response
    )
    escaped = prompt.replace("'", "''")
    result = session.sql(
        f"SELECT SNOWFLAKE.CORTEX.COMPLETE('{JUDGE_MODEL}', '{escaped}') AS resp"
    ).collect()
    raw = result[0]["RESP"]
    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        match = re.search(r'\{[^}]+\}', raw)
        if match:
            return json.loads(match.group())
        return {"answer_correctness": 0, "reasoning_quality": 0, "reasoning": "parse_error"}


# Score baseline
print("Scoring Cortex baseline responses...")
for i, r in enumerate(baseline_results):
    scores = llm_judge(r["question"], r["ground_truth"], r["response"])
    r.update(scores)
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(baseline_results)}]")

# Score fine-tuned
print("Scoring Tinker fine-tuned responses...")
for i, r in enumerate(finetuned_results):
    scores = llm_judge(r["question"], r["ground_truth"], r["response"])
    r.update(scores)
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(finetuned_results)}]")

# Summary
b_corr = np.mean([r["answer_correctness"] for r in baseline_results])
b_reas = np.mean([r["reasoning_quality"] for r in baseline_results])
f_corr = np.mean([r["answer_correctness"] for r in finetuned_results])
f_reas = np.mean([r["reasoning_quality"] for r in finetuned_results])

print(f"\n{'Metric':<25} {'Cortex Base':>12} {'Tinker RL':>12} {'Delta':>8}")
print("-" * 58)
print(f"{'Answer Correctness /5':<25} {b_corr:>12.2f} {f_corr:>12.2f} {f_corr-b_corr:>+8.2f}")
print(f"{'Reasoning Quality  /5':<25} {b_reas:>12.2f} {f_reas:>12.2f} {f_reas-b_reas:>+8.2f}")
print(f"{'Exact Match Accuracy':<25} {baseline_acc:>12.1%} {finetuned_acc:>12.1%} {finetuned_acc-baseline_acc:>+8.1%}")

In [ ]:
# ── Visualize LLM Judge Scores ──
import matplotlib.pyplot as plt

metrics = ["Answer\nCorrectness", "Reasoning\nQuality", "Exact Match\nAccuracy"]
base_scores = [b_corr / 5, b_reas / 5, baseline_acc]     # normalize to 0-1
ft_scores   = [f_corr / 5, f_reas / 5, finetuned_acc]

x = np.arange(len(metrics))
w = 0.3

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - w/2, base_scores, w, label=f"Cortex {CORTEX_MODEL} (base)", color="#29B5E8")
bars2 = ax.bar(x + w/2, ft_scores,   w, label=f"Tinker {TINKER_MODEL} (RL)",   color="#FF6F00")

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{bar.get_height():.0%}", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel("Score (normalized 0-1)")
ax.set_title("Cortex Agent Evaluations — LLM Judge Scores")
ax.set_ylim(0, 1.2)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Cortex AI Evaluation Setup
Prepare a ground-truth dataset and evaluation config YAML for use with `EXECUTE_AI_EVALUATION`.

In [ ]:
# ── Prepare ground truth dataset for EXECUTE_AI_EVALUATION ──

eval_rows = []
for row in test_data:
    gt = extract_gsm8k_answer(row["answer"])
    eval_rows.append({
        "INPUT_QUERY": row["question"] + " Put your final answer inside \\boxed{}.",
        "GROUND_TRUTH": json.dumps({
            "ground_truth_output": f"The answer is {gt}.",
            "ground_truth_invocations": []
        }),
    })

eval_df = session.create_dataframe(pd.DataFrame(eval_rows))
eval_df.write.mode("overwrite").save_as_table("MATH_EVAL_GROUND_TRUTH")
print(f"Ground truth dataset: {len(eval_rows)} rows -> MATH_EVAL_GROUND_TRUTH")

In [ ]:
# ── Stage eval config YAML with custom math reasoning metric ──

eval_config_yaml = """
version: "1.0"
agent:
  name: "<YOUR_DB>.<YOUR_SCHEMA>.<YOUR_MATH_AGENT>"
  type: "cortex agent"
dataset:
  table: "MATH_EVAL_GROUND_TRUTH"
  input_column: "INPUT_QUERY"
  ground_truth_column: "GROUND_TRUTH"
metrics:
  - name: "answer_correctness"
    type: "builtin"
  - name: "logical_consistency"
    type: "builtin"
  - name: "math_reasoning"
    type: "custom"
    prompt: |
      Evaluate the mathematical reasoning in the agent response.
      Check: (1) correct arithmetic, (2) logical step progression,
      (3) final answer matches work shown, (4) no skipped steps.
      Score 1-5 where 5 is flawless reasoning.
    score_range: [1, 5]
""".strip()

# Write config to stage
session.sql("CREATE STAGE IF NOT EXISTS EVAL_CONFIGS ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')").collect()

import tempfile, os
with tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False) as f:
    f.write(eval_config_yaml)
    tmp_path = f.name

session.file.put(f"file://{tmp_path}", "@EVAL_CONFIGS", auto_compress=False, overwrite=True)
os.unlink(tmp_path)
print("Eval config staged at @EVAL_CONFIGS/")
print("\nTo run against your deployed Cortex Agent:")
print("""CALL EXECUTE_AI_EVALUATION(
    'START',
    OBJECT_CONSTRUCT('run_name', 'math-eval-run-1'),
    '@EVAL_CONFIGS/eval_config.yaml'
);""")

## 6. Save Results to Snowflake
Persist the LLM judge scores and comparison data to Snowflake tables for downstream analysis.

In [ ]:
# ── Save LLM judge scores to Snowflake ──

judge_comparison = []
for b, f in zip(baseline_results, finetuned_results):
    judge_comparison.append({
        "QUESTION":                b["question"],
        "GROUND_TRUTH":            b["ground_truth"],
        "CORTEX_ANSWER":           b["model_answer"],
        "CORTEX_CORRECT":          b["correct"],
        "CORTEX_JUDGE_CORRECTNESS": b.get("answer_correctness", 0),
        "CORTEX_JUDGE_REASONING":  b.get("reasoning_quality", 0),
        "TINKER_ANSWER":           f["model_answer"],
        "TINKER_CORRECT":          f["correct"],
        "TINKER_JUDGE_CORRECTNESS": f.get("answer_correctness", 0),
        "TINKER_JUDGE_REASONING":  f.get("reasoning_quality", 0),
    })

judge_df = session.create_dataframe(pd.DataFrame(judge_comparison))
judge_df.write.mode("overwrite").save_as_table("TINKER_VS_CORTEX_JUDGE_SCORES")
print("LLM judge scores saved to TINKER_VS_CORTEX_JUDGE_SCORES")

## 7. Comprehensive Judge Dashboard
A multi-panel visualization comparing Tinker RL and Cortex across all judge dimensions: head-to-head wins, mean scores, distributions, per-problem breakdowns, and summary statistics.

In [ ]:
# ── Comprehensive Judge Dashboard ──
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import numpy as np

jdf = judge_df.to_pandas()

# Derived columns
jdf["CORTEX_JUDGE_TOTAL"] = jdf["CORTEX_JUDGE_CORRECTNESS"] + jdf["CORTEX_JUDGE_REASONING"]
jdf["TINKER_JUDGE_TOTAL"]  = jdf["TINKER_JUDGE_CORRECTNESS"] + jdf["TINKER_JUDGE_REASONING"]
jdf["TINKER_WINS"]  = jdf["TINKER_JUDGE_TOTAL"] > jdf["CORTEX_JUDGE_TOTAL"]
jdf["CORTEX_WINS"]  = jdf["CORTEX_JUDGE_TOTAL"] > jdf["TINKER_JUDGE_TOTAL"]
jdf["TIES"]         = jdf["TINKER_JUDGE_TOTAL"] == jdf["CORTEX_JUDGE_TOTAL"]

# Color palette
CORTEX_CLR = "#5A9BD5"
TINKER_CLR = "#ED7D31"
TIE_CLR    = "#A5A5A5"
BG_CLR     = "#FAFAFA"

fig = plt.figure(figsize=(18, 16), facecolor=BG_CLR)
fig.suptitle("Tinker RL vs Cortex Baseline  —  LLM Judge Evaluation",
             fontsize=20, fontweight="bold", y=0.98, color="#2B2B2B")

gs = gridspec.GridSpec(3, 3, hspace=0.45, wspace=0.35,
                       left=0.06, right=0.96, top=0.93, bottom=0.04)

# ─── 1. Win / Tie / Loss donut ───
ax1 = fig.add_subplot(gs[0, 0])
wins  = jdf["TINKER_WINS"].sum()
losses = jdf["CORTEX_WINS"].sum()
ties  = jdf["TIES"].sum()
sizes = [wins, ties, losses]
lbls  = [f"Tinker wins\n{wins}", f"Tie\n{ties}", f"Cortex wins\n{losses}"]
wedges, texts, autotexts = ax1.pie(
    sizes, labels=lbls, colors=[TINKER_CLR, TIE_CLR, CORTEX_CLR],
    autopct="%1.0f%%", startangle=90, pctdistance=0.78,
    wedgeprops=dict(width=0.45, edgecolor="white", linewidth=2))
for t in autotexts:
    t.set_fontsize(10)
    t.set_fontweight("bold")
ax1.set_title("Head-to-Head (Combined Score)", fontsize=12, fontweight="bold", pad=12)

# ─── 2. Mean scores grouped bar ───
ax2 = fig.add_subplot(gs[0, 1])
metrics = ["Correctness", "Reasoning", "Combined"]
cortex_means = [
    jdf["CORTEX_JUDGE_CORRECTNESS"].mean(),
    jdf["CORTEX_JUDGE_REASONING"].mean(),
    jdf["CORTEX_JUDGE_TOTAL"].mean() / 2,
]
tinker_means = [
    jdf["TINKER_JUDGE_CORRECTNESS"].mean(),
    jdf["TINKER_JUDGE_REASONING"].mean(),
    jdf["TINKER_JUDGE_TOTAL"].mean() / 2,
]
x = np.arange(len(metrics))
w = 0.32
b1 = ax2.bar(x - w/2, cortex_means, w, label="Cortex", color=CORTEX_CLR, edgecolor="white", zorder=3)
b2 = ax2.bar(x + w/2, tinker_means, w, label="Tinker RL", color=TINKER_CLR, edgecolor="white", zorder=3)
ax2.set_xticks(x)
ax2.set_xticklabels(metrics)
ax2.set_ylim(0, 5.5)
ax2.set_ylabel("Mean Score (0-5)")
ax2.bar_label(b1, fmt="%.2f", fontsize=9, padding=2)
ax2.bar_label(b2, fmt="%.2f", fontsize=9, padding=2)
ax2.legend(frameon=False, fontsize=9)
ax2.set_title("Mean Judge Scores", fontsize=12, fontweight="bold", pad=12)
ax2.grid(axis="y", alpha=0.3, zorder=0)
ax2.set_axisbelow(True)

# ─── 3. Score distribution histograms (overlapping) ───
ax3 = fig.add_subplot(gs[0, 2])
bins = np.arange(-0.5, 11, 1)
ax3.hist(jdf["CORTEX_JUDGE_TOTAL"], bins=bins, alpha=0.55, color=CORTEX_CLR,
         label="Cortex", edgecolor="white", linewidth=0.8)
ax3.hist(jdf["TINKER_JUDGE_TOTAL"], bins=bins, alpha=0.55, color=TINKER_CLR,
         label="Tinker RL", edgecolor="white", linewidth=0.8)
ax3.set_xlabel("Combined Judge Score (0-10)")
ax3.set_ylabel("# Problems")
ax3.legend(frameon=False, fontsize=9)
ax3.set_title("Score Distribution", fontsize=12, fontweight="bold", pad=12)
ax3.grid(axis="y", alpha=0.3)

# ─── 4. Per-question paired dot plot (correctness) ───
ax4 = fig.add_subplot(gs[1, :])
idx = np.arange(len(jdf))
ax4.scatter(idx, jdf["CORTEX_JUDGE_CORRECTNESS"], s=40, c=CORTEX_CLR,
            alpha=0.7, label="Cortex", zorder=3, marker="o")
ax4.scatter(idx, jdf["TINKER_JUDGE_CORRECTNESS"], s=40, c=TINKER_CLR,
            alpha=0.7, label="Tinker RL", zorder=3, marker="D")
for i in idx:
    ax4.plot([i, i],
             [jdf["CORTEX_JUDGE_CORRECTNESS"].iloc[i],
              jdf["TINKER_JUDGE_CORRECTNESS"].iloc[i]],
             color="#CCCCCC", linewidth=0.6, zorder=1)
ax4.set_xlabel("Problem Index")
ax4.set_ylabel("Correctness Score (0-5)")
ax4.set_ylim(-0.3, 5.5)
ax4.legend(frameon=False, fontsize=9, loc="lower right")
ax4.set_title("Per-Problem Correctness  (connected dots = same problem)",
              fontsize=12, fontweight="bold", pad=12)
ax4.grid(axis="y", alpha=0.3)

# ─── 5. Correctness score heatmap (Cortex vs Tinker) ───
ax5 = fig.add_subplot(gs[2, 0])
from matplotlib.colors import LinearSegmentedColormap
heat = np.zeros((6, 6))
for _, row in jdf.iterrows():
    c = int(row["CORTEX_JUDGE_CORRECTNESS"])
    t = int(row["TINKER_JUDGE_CORRECTNESS"])
    heat[c, t] += 1
cmap = LinearSegmentedColormap.from_list("custom", ["#FFFFFF", TINKER_CLR])
im = ax5.imshow(heat, cmap=cmap, origin="lower", aspect="equal")
for i in range(6):
    for j in range(6):
        val = int(heat[i, j])
        if val > 0:
            ax5.text(j, i, str(val), ha="center", va="center",
                     fontsize=10, fontweight="bold",
                     color="white" if val > heat.max() * 0.6 else "#333")
ax5.set_xticks(range(6))
ax5.set_yticks(range(6))
ax5.set_xlabel("Tinker Correctness")
ax5.set_ylabel("Cortex Correctness")
ax5.set_title("Correctness Confusion", fontsize=12, fontweight="bold", pad=12)

# ─── 6. Delta plot (Tinker - Cortex combined) ───
ax6 = fig.add_subplot(gs[2, 1])
delta = jdf["TINKER_JUDGE_TOTAL"] - jdf["CORTEX_JUDGE_TOTAL"]
colors_delta = [TINKER_CLR if d > 0 else CORTEX_CLR if d < 0 else TIE_CLR for d in delta]
sorted_idx = delta.sort_values().index
ax6.barh(range(len(delta)), delta.loc[sorted_idx].values,
         color=[colors_delta[i] for i in sorted_idx],
         edgecolor="white", linewidth=0.3)
ax6.axvline(0, color="#333", linewidth=0.8)
ax6.set_xlabel("Score Delta (Tinker - Cortex)")
ax6.set_ylabel("Problems (sorted)")
ax6.set_title("Per-Problem Delta", fontsize=12, fontweight="bold", pad=12)
ax6.grid(axis="x", alpha=0.3)

# ─── 7. Summary stats table ───
ax7 = fig.add_subplot(gs[2, 2])
ax7.axis("off")
cortex_acc = jdf["CORTEX_CORRECT"].mean()
tinker_acc = jdf["TINKER_CORRECT"].mean()
rows = [
    ["Exact Match Acc",  f"{cortex_acc:.0%}",  f"{tinker_acc:.0%}",  f"{tinker_acc - cortex_acc:+.0%}"],
    ["Correctness /5",   f"{jdf['CORTEX_JUDGE_CORRECTNESS'].mean():.2f}",
                         f"{jdf['TINKER_JUDGE_CORRECTNESS'].mean():.2f}",
                         f"{jdf['TINKER_JUDGE_CORRECTNESS'].mean() - jdf['CORTEX_JUDGE_CORRECTNESS'].mean():+.2f}"],
    ["Reasoning /5",     f"{jdf['CORTEX_JUDGE_REASONING'].mean():.2f}",
                         f"{jdf['TINKER_JUDGE_REASONING'].mean():.2f}",
                         f"{jdf['TINKER_JUDGE_REASONING'].mean() - jdf['CORTEX_JUDGE_REASONING'].mean():+.2f}"],
    ["Perfect 10s",      str((jdf['CORTEX_JUDGE_TOTAL'] == 10).sum()),
                         str((jdf['TINKER_JUDGE_TOTAL'] == 10).sum()), ""],
    ["Zero Scores",      str((jdf['CORTEX_JUDGE_TOTAL'] == 0).sum()),
                         str((jdf['TINKER_JUDGE_TOTAL'] == 0).sum()), ""],
]
table = ax7.table(cellText=rows,
                  colLabels=["Metric", "Cortex", "Tinker RL", "Delta"],
                  loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 1.6)
for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor("#DDDDDD")
    if r == 0:
        cell.set_facecolor("#2B2B2B")
        cell.set_text_props(color="white", fontweight="bold")
    elif c == 3:
        txt = cell.get_text().get_text()
        if txt.startswith("+"):
            cell.set_text_props(color="#2E7D32", fontweight="bold")
        elif txt.startswith("-"):
            cell.set_text_props(color="#C62828", fontweight="bold")
ax7.set_title("Summary", fontsize=12, fontweight="bold", pad=12)

plt.show()